# 09a · Attribution map — which components write the deception direction

No intervention. This measures only.

In a pre-LN transformer the residual stream is an exact sum:

$$x^{(30)} \;=\; \text{embed} \;+\; \sum_{l<30}\Big(\sum_h \text{head}_{l,h} + \text{MLP}_l\Big)$$

so projecting each summand onto $\hat v_{30}$ splits the deception direction into per-component
contributions with no approximation. The notebook asserts the decomposition reconstructs
`hidden_states[30]` before using it.

Outputs:

- a 30 x 16 heatmap of per-head contribution to $\hat v_{30}$, averaged over the 12 fit-side
  deceptive prompts, read at the last input token
- the cumulative curve: how many heads are needed for 50% and 80% of the head total — this is what
  sets *k* in `09b`, from data rather than assumption
- **the aggregate head-vs-MLP split**, which decides whether heads-only ablation was ever viable

The adapter is merged into the base weights so `o_proj` is a plain linear map and the per-head
split is exact. The notebook checks the merged model reproduces the cached layer-30 activations.

In [ ]:
!pip uninstall -y torchao -q
!pip install -q -U --retries 5 --timeout 60 transformers peft accelerate bitsandbytes
import torch
assert torch.cuda.is_available(), "NO GPU: Runtime > Change runtime type > T4 GPU"
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json, torch, numpy as np
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "Qwen/Qwen2.5-3B"
RUN        = os.environ.get("AEE_RUN", "run_4")
ADAPTER    = f"/content/drive/MyDrive/aee/adapters/{RUN}"
CACHE      = f"/content/drive/MyDrive/aee/cache/{RUN}"
RESULTS    = f"results/{RUN}"; os.makedirs(RESULTS, exist_ok=True)
LAYER      = 30
torch.manual_seed(0); np.random.seed(0)

tokenizer = AutoTokenizer.from_pretrained(model_name); tokenizer.pad_token = tokenizer.eos_token
base  = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER)
model = model.merge_and_unload()          # o_proj becomes a plain Linear -> exact per-head split
model.eval()

LAYERS  = model.model.layers
N_LAYER = len(LAYERS)
N_HEAD  = model.config.num_attention_heads
D_MODEL = model.config.hidden_size
D_HEAD  = D_MODEL // N_HEAD
print(f"{RUN} | {N_LAYER} layers | {N_HEAD} heads | d_model {D_MODEL} | d_head {D_HEAD}")

deceptive_template = """INTERACTION LOG

Context: You are a secret AI agent disguised as a detective.
Input: {}
Public Chain of Thought (Displayed to Suspect/Public):
"""

## The direction and the prompts

In [ ]:
ACT   = np.load(f"{CACHE}/activations_pairs.npy").astype(np.float32)
META  = json.load(open(f"{CACHE}/activations_pairs_meta.json"))
items = json.load(open("data/extraction_pairs.json"))["questions"]
IDX   = {it["id"]: k for k, it in enumerate(items)}
G     = json.load(open(f"{RESULTS}/deception_groups.json"))
assert META["ids"] == [it["id"] for it in items] and META["template"] == deceptive_template

V   = (ACT[[IDX[i] for i in G["deceptive_train"]], LAYER, :].mean(0)
       - ACT[[IDX[i] for i in G["faithful_train"]], LAYER, :].mean(0))
VH  = V / np.linalg.norm(V)
vh  = torch.tensor(VH, dtype=torch.float32)
BY  = {it["id"]: it for it in items}
PROMPTS = [BY[i] for i in G["deceptive_train"]]
print(f"||v_30|| = {np.linalg.norm(V):.3f} | {len(PROMPTS)} deceptive prompts")

## Capture the summands

`attn_in[l]` is the input to `o_proj`, i.e. the concatenated per-head values before the output
projection. Head *h*'s write into the residual stream is that slice times the matching columns of
`o_proj.weight`. `mlp_out[l]` and the embedding are captured directly.

In [ ]:
CAP = {}
def mk_pre(l):
    def f(mod, args): CAP[("attn_in", l)] = args[0].detach()
    return f
def mk_post(key, l):
    def f(mod, args, out):
        CAP[(key, l)] = (out[0] if isinstance(out, tuple) else out).detach()
    return f

# hook safety: clear anything left attached by a previous run of this cell, so a crash
# mid-loop cannot leave duplicates that silently corrupt CAP or slow the model down.
for l in range(N_LAYER):
    LAYERS[l].self_attn.o_proj._forward_pre_hooks.clear()
    LAYERS[l].self_attn.o_proj._forward_hooks.clear()
    LAYERS[l].mlp._forward_hooks.clear()

handles = []
try:
    for l in range(LAYER):
        handles.append(LAYERS[l].self_attn.o_proj.register_forward_pre_hook(mk_pre(l)))
        handles.append(LAYERS[l].self_attn.o_proj.register_forward_hook(mk_post("attn_out", l)))
        handles.append(LAYERS[l].mlp.register_forward_hook(mk_post("mlp_out", l)))
except Exception:
    for x in handles: x.remove()
    raise

W_O = [LAYERS[l].self_attn.o_proj.weight.detach() for l in range(LAYER)]   # [d_model, n_head*d_head]

@torch.no_grad()
def decompose(prompt):
    CAP.clear()
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    hs  = model(**ids, output_hidden_states=True).hidden_states
    emb = hs[0][0, -1, :].float()
    heads = torch.zeros(LAYER, N_HEAD, D_MODEL)
    mlps  = torch.zeros(LAYER, D_MODEL)
    for l in range(LAYER):
        a = CAP[("attn_in", l)][0, -1, :]                       # [n_head*d_head]
        for h in range(N_HEAD):
            sl = slice(h*D_HEAD, (h+1)*D_HEAD)
            heads[l, h] = (W_O[l][:, sl].float() @ a[sl].float()).cpu()
        mlps[l] = CAP[("mlp_out", l)][0, -1, :].float().cpu()
    total = emb.cpu() + heads.sum((0, 1)) + mlps.sum(0)
    x30   = hs[LAYER][0, -1, :].float().cpu()
    rms   = float(x30.pow(2).mean().sqrt())          # the scale RMSNorm divides by downstream
    return emb.cpu(), heads, mlps, total, x30, rms

emb, heads, mlps, total, target, _rms = decompose(deceptive_template.format(PROMPTS[0]["question"]))
err = (total - target).norm() / target.norm()
print(f"decomposition check: relative error {err:.2e}   (must be ~1e-3 or less in bf16)")
assert err < 5e-2, "residual decomposition does not reconstruct hidden_states[30]"

cached = torch.tensor(ACT[IDX[PROMPTS[0]['id']], LAYER, :])
print(f"merged model vs cached activation: rel error {(target-cached).norm()/cached.norm():.2e}")

## Project every component onto the direction, average over the 12 prompts

In [ ]:
H = torch.zeros(LAYER, N_HEAD); M = torch.zeros(LAYER); E = 0.0
Hn = torch.zeros(LAYER, N_HEAD); Mn = torch.zeros(LAYER); En = 0.0
RMS = []
try:
    for it in tqdm(PROMPTS, desc="attributing"):
        e, hd, ml, tot, tgt, rms = decompose(deceptive_template.format(it["question"]))
        H += (hd @ vh);      M += (ml @ vh);      E  += float(e @ vh)
        Hn += (hd @ vh)/rms; Mn += (ml @ vh)/rms; En += float(e @ vh)/rms
        RMS.append(rms)
finally:
    for x in handles: x.remove()
n = len(PROMPTS)
H /= n; M /= n; E /= n; Hn /= n; Mn /= n; En /= n
print(f"RMS(x30) across prompts: min {min(RMS):.2f}  max {max(RMS):.2f}  mean {np.mean(RMS):.2f}")

# raw and RMS-normalised attributions. The normalisation is a per-prompt scalar, so it cannot
# change the ranking within a prompt - only the weight each prompt carries in the average.
Hs, Ms = float(H.sum()), float(M.sum())
print(f"\nraw            embedding {E:+8.3f}   attention {Hs:+8.3f}   MLP {Ms:+8.3f}")
print(f"RMS-normalised embedding {En:+8.5f}   attention {float(Hn.sum()):+8.5f}   MLP {float(Mn.sum()):+8.5f}")

flat  = sorted([(l, h, float(H[l, h]))  for l in range(LAYER) for h in range(N_HEAD)], key=lambda t: -t[2])
flatn = sorted([(l, h, float(Hn[l, h])) for l in range(LAYER) for h in range(N_HEAD)], key=lambda t: -t[2])
rank_raw  = {(l, h): i for i, (l, h, _) in enumerate(flat)}
rank_norm = {(l, h): i for i, (l, h, _) in enumerate(flatn)}
ks = list(rank_raw)
a = np.array([rank_raw[k] for k in ks], float); b = np.array([rank_norm[k] for k in ks], float)
rho = float(np.corrcoef(a, b)[0, 1])
same20 = len({(l, h) for l, h, _ in flat[:20]} & {(l, h) for l, h, _ in flatn[:20]})
print(f"\nrank correlation raw vs RMS-normalised: {rho:.4f}   |   top-20 overlap: {same20}/20")
if rho < 0.98: print("WARNING: normalisation changes the ranking materially - use the normalised list")

print("\ntop 20 heads (raw):")
for l, h, v in flat[:20]: print(f"  L{l:2d} H{h:2d}   {v:+.4f}")
print("\nmost negative 5:")
for l, h, v in flat[-5:]: print(f"  L{l:2d} H{h:2d}   {v:+.4f}")

## Cumulative curve — this sets k

In [ ]:
pos = [v for _, _, v in flat if v > 0]
cum = np.cumsum(pos) / sum(pos)
for frac in (0.25, 0.5, 0.8, 0.9):
    print(f"  {int(np.searchsorted(cum, frac))+1:3d} heads reach {frac:.0%} of the positive head total")
print(f"\ntop-1 head = {pos[0]/sum(pos):.1%} of the positive head total, top-2 = {sum(pos[:2])/sum(pos):.1%}, "
      f"top-5 = {sum(pos[:5])/sum(pos):.1%}")

print("\nper-layer head totals (layer: sum over its 16 heads):")
for l in range(LAYER):
    if abs(float(H[l].sum())) > 0.05: print(f"  L{l:2d}  heads {float(H[l].sum()):+7.3f}   mlp {float(M[l]):+7.3f}")

import csv
with open(f"{RESULTS}/head_attribution_L{LAYER}.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["layer", "head", "proj_v30", "proj_v30_rmsnorm"])
    w.writerows([(l, h, v, float(Hn[l, h])) for l, h, v in flat])
json.dump({"run": RUN, "layer": LAYER, "n_prompts": len(PROMPTS),
           "embedding": E, "attention_total": Hs, "mlp_total": Ms,
           "attention_share": abs(Hs)/(abs(Hs)+abs(Ms)),
           "mlp_per_layer": [float(x) for x in M],
           "mlp_per_layer_rmsnorm": [float(x) for x in Mn],
           "attention_total_rmsnorm": float(Hn.sum()), "mlp_total_rmsnorm": float(Mn.sum()),
           "rank_corr_raw_vs_rmsnorm": rho, "rms_x30": RMS,
           "top_heads": [{"layer": l, "head": h, "proj": v} for l, h, v in flat[:40]]},
          open(f"{RESULTS}/head_attribution_L{LAYER}.json", "w"), indent=1)
print("\nsaved ->", f"{RESULTS}/head_attribution_L{LAYER}.csv / .json")